In [1]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# Production layout: add project root and src (run from repo root or notebooks/ingestion/)
_root = Path(".").resolve()
if _root.name == "ingestion":
    _root = _root.parent.parent
elif (_root / "src").is_dir():
    pass
else:
    _root = _root.parent
load_dotenv(_root / ".env")
load_dotenv("/app/.env")
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_root / "src"))

from storage.postgres.pgConn import PgConn
from storage.postgres import PostgresSQL_table_queries
from storage.cloud.CloudStorage import CloudStorageProvider

import pandas as pd
from datetime import datetime

In [2]:
pg_conn = PgConn("historical")
df = pg_conn.get_stocks_prices()
if df is None:
    raise RuntimeError(
        "get_stocks_prices() failed — check the error printed above "
        "(connection, table name, or schema)."
    )

Connection to the database successful!
Table name set to: historical
Error: Unable to retrieve data from the database. 9 columns passed, passed data had 10 columns


In [3]:
df.head()

AttributeError: 'NoneType' object has no attribute 'head'

In [ ]:
df.count()

In [ ]:
# Assuming df is your DataFrame
date_column_type = df['date'].dtype
print("Type of values in 'date' column:", date_column_type)

In [ ]:
class DataETL():
    
    def __init__(self, dataframe):
            self.df = dataframe
    
    class Export():
        def __init__(self, dataframe):
            self.cloudProvider = CloudStorageProvider()
            self.df = dataframe
            
        def export_stocks_to_s3(self, bucket_name, prefix_path, file_format):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()

            # Create a new bucket
            aws_storage.create_bucket(bucket_name)

            # Upload DataFrame with datetime subfolder structure
            aws_storage.upload_dataframe_with_timestamp(self.df, bucket_name, prefix_path, file_format)
            
        def export_stocks_to_s3_full_file(self, bucket_name, prefix_path, filename):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()
            aws_storage.upload_dataframe_to_csv(self.df, bucket_name, filename, prefix_path)
            
    class Ingestion():
        
        def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
        def get_full_data_csv_file(self, bucket_name, prefix_path):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()
            return aws_storage.get_csv_from_specific_folder(bucket_name, prefix_path)
    
    class Process():
        
        def __init__(self, dataframe):
            self.df = dataframe
        
        def getData():
            self.df = pg_conn.get_financial_news()
        
        def filter_by_current_date(self):
            today = datetime.today().date()
            dates = pd.to_datetime(self.df["date"], errors="coerce")
            return self.df[dates.dt.date == today]
        
        def filter_by_date_range(self, df, start_date, end_date):
            # Convert start_date and end_date strings to datetime objects
            start_date = pd.to_datetime(start_date)
            end_date = pd.to_datetime(end_date)
            
            df['date'] = pd.to_datetime(df['date'])
            # Filter by date range
            filtered_df = df[(df['date'] >= start_date) & (df['date'] <= end_date)]
            return filtered_df
        
        def filter_by_book(self, df, book=None):
            # Filter by book if specified
            if book is not None:
                filtered_df = df[df['book'] == book]
            return filtered_df
        
        def get_unique_by_date(self, df):
            # Sort the DataFrame by 'date' in descending order
            df_sorted = df.sort_values(by='date', ascending=False)

            # Drop duplicates, keeping only the first occurrence for each unique combination of book and date
            df_unique_latest = df_sorted.drop_duplicates(subset=['book', 'date'])
            df_unique_latest.count()
            return df_unique_latest
            
    class Transform():
        def extractStopWords():
            pass

In [ ]:
FILTER_BY_CURRENT_DATE = False
FILTER_BY_RANGE_DATE = True
FILTER_BY_BOOK_DATE = False
# TODO 2017-2016
etl = DataETL(df)
etl_process = etl.Process(etl.df)
uniqued_df = etl_process.get_unique_by_date(etl_process.df)
processed_df = uniqued_df

if FILTER_BY_CURRENT_DATE == True:
    processed_df = etl_process.filter_by_current_date(uniqued_df)
    processed_df.head()
elif FILTER_BY_RANGE_DATE == True:
    start_date='2026-05-17'
    end_date='2026-05-24'
    processed_df = etl_process.filter_by_date_range(uniqued_df, start_date, end_date)
    processed_df.head()
elif FILTER_BY_BOOK_DATE == True:
    processed_df = etl_process.filter_by_current_date(uniqued_df)
    processed_df.head()

In [ ]:
etl_export = etl.Export(processed_df)
bucket_name = "test-financial-stocks-bucket"
prefix_path = "stocks/crypto"
post_full_csv = False
post_to_s3 = True
now = datetime.now()
filename = f"{now.year}-{now.month:02}-{now.day:02}_full_record"
file_format = "csv"
if post_to_s3 == True:
    etl_export.export_stocks_to_s3(bucket_name, prefix_path, file_format)
if post_full_csv == True:
    etl_export.export_stocks_to_s3_full_file(bucket_name, prefix_path, filename)

In [ ]:
ingest_data = False
get_full_file = True
df_from_file = None

if ingest_data == True:
    etl_ingestion = etl.Ingestion(etl.df)
    bucket_name = "test-financial-stocks-bucket"
    prefix_path = "stocks/crypto/"
    year = ''
    mont = ''
    day = ''
    hour = ''
    minute = ''
    if get_full_file == True:
        df_from_file = etl_ingestion.get_full_data_csv_file(bucket_name, prefix_path)

In [ ]:
if df_from_file is not None:
    print(df_from_file.head())